# 00 — Verify connection & lay the foundation

**basketball-reference-waf** is an end-to-end Databricks demo. It ingests NBA data from
[Basketball Reference](https://www.basketball-reference.com/), runs it through a
**bronze → silver → gold** medallion ("WAF") pipeline, and trains a simple ML model that
predicts the **home team's win probability** for each game.

This first notebook only sets up the workspace:

| Step | What & why (Well-Architected Framework) |
|------|------------------------------------------|
| Serverless connection | `DatabricksSession.builder.serverless()` — no cluster to manage (**Cost / Operational Excellence**) |
| Catalog + 3 schemas | `_bronze` / `_silver` / `_gold` separate raw, cleaned, and serving data (**Reliability**) |
| Landing Volume | A UC Volume for raw scraped JSON — governed object storage (**Security / Governance**) |
| MLflow experiment | Tracking + UC model-registry URIs, so later notebooks log runs and register models |

> Run notebooks `00` → `08` in order; each depends on tables the previous one created.

In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()

me = w.current_user.me()
print("Connected as:", me.user_name)
print("Spark version:", spark.version)

/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:141: UserWarning: Could not enable SetAllowOversizeProtos, please check the protobuf version.
  warnings.warn("Could not enable SetAllowOversizeProtos, please check the protobuf version.")


Connected as: alexander.booth@databricks.com
Spark version: 4.1.0


## Configuration

Everything is parameterized from `.env`, so the same notebooks run as a quick demo or a
full multi-season backfill.

In [2]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "basketball_reference_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

MODEL_NAME      = os.getenv("MODEL_NAME", "home_win_classifier")
FULL_MODEL_NAME = f"{UC_CATALOG}.{GOLD_SCHEMA}.{MODEL_NAME}"
EXPERIMENT      = os.getenv("MLFLOW_EXPERIMENT_NAME", f"/Users/{me.user_name}/basketball-reference-waf")
SEASONS         = [s.strip() for s in os.getenv("NBA_SEASONS", "2023,2024,2025").split(",") if s.strip()]

print(f"Catalog          : {UC_CATALOG}")
print(f"Schemas          : {BRONZE_SCHEMA} | {SILVER_SCHEMA} | {GOLD_SCHEMA}")
print(f"Landing Volume   : {VOLUME_PATH}")
print(f"Model (UC)       : {FULL_MODEL_NAME}")
print(f"MLflow experiment: {EXPERIMENT}")
print(f"Seasons          : {SEASONS}")

Catalog          : alexander_booth
Schemas          : basketball_reference_waf_bronze | basketball_reference_waf_silver | basketball_reference_waf_gold
Landing Volume   : /Volumes/alexander_booth/basketball_reference_waf_bronze/raw_data
Model (UC)       : alexander_booth.basketball_reference_waf_gold.home_win_classifier
MLflow experiment: /Users/alexander.booth@databricks.com/basketball-reference-waf
Seasons          : ['2023', '2024', '2025']


## Create the catalog, schemas, and landing Volume

`IF NOT EXISTS` everywhere keeps this notebook **idempotent** — safe to re-run any time.

In [3]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {UC_CATALOG}")

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}
    COMMENT 'Bronze - raw Basketball Reference JSON (schedule + team box scores) landed via Auto Loader, stored as VARIANT.'
""")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{SILVER_SCHEMA}
    COMMENT 'Silver - typed, deduped games and team-game box scores with computed Four Factors.'
""")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{GOLD_SCHEMA}
    COMMENT 'Gold - star schema (dims + facts), ML feature table, and the home-win model + predictions.'
""")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

print("Catalog, schemas, and landing Volume are ready.\n")
display_rows = spark.sql(f"SHOW SCHEMAS IN {UC_CATALOG} LIKE '{UC_SCHEMA}*'").collect()
for row in display_rows:
    print("  schema:", row[0])

Catalog, schemas, and landing Volume are ready.



  schema: basketball_reference_waf_bronze
  schema: basketball_reference_waf_gold
  schema: basketball_reference_waf_silver


## MLflow — tracking + Unity Catalog model registry

Point MLflow at Databricks for experiment tracking and at **Unity Catalog** for the model
registry (so the model we train later is governed like any other UC asset).

In [4]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
exp = mlflow.set_experiment(EXPERIMENT)

print("MLflow experiment ready:")
print("  name:", exp.name)
print("  id  :", exp.experiment_id)

2026/06/02 11:57:23 INFO mlflow.tracking.fluent: Experiment with name '/Users/alexander.booth@databricks.com/basketball-reference-waf' does not exist. Creating a new experiment.


MLflow experiment ready:
  name: /Users/alexander.booth@databricks.com/basketball-reference-waf
  id  : 112419715817170


Foundation is in place. **Next:** `01_ingest_basketball_reference` scrapes the season
schedule and per-game team box scores into the landing Volume.